In [0]:
# Create a JSON file with bad records
import json

mixed_data = [
    {"id": 1, "customer": "Alice", "amount": 1500.00},
    {"id": "BAD", "customer": "Bob", "amount": "not_a_number"},  # ← bad!
    {"id": 3, "customer": "Charlie", "amount": 5000.00},
    {"id": None, "customer": "Diana", "amount": 450.00}  # ← null id!
]

dbutils.fs.put(
    "/Volumes/workspace/default/raw_transactions/bad_records.json",
    "\n".join([json.dumps(r) for r in mixed_data]),
    overwrite=True
)

print("✅ File with bad records created!")

In [0]:
# Save to Delta table first then query
df_permissive.write.format("delta").mode("overwrite").saveAsTable("temp_permissive")

print("=== PERMISSIVE MODE ===")
spark.sql("SELECT * FROM temp_permissive").show(truncate=False)
print(f"Total records: {spark.sql('SELECT COUNT(*) FROM temp_permissive').collect()[0][0]}")
print(f"Bad records: {spark.sql('SELECT COUNT(*) FROM temp_permissive WHERE _corrupt_record IS NOT NULL').collect()[0][0]}")
print(f"Good records: {spark.sql('SELECT COUNT(*) FROM temp_permissive WHERE _corrupt_record IS NULL').collect()[0][0]}")

In [0]:
# DROPMALFORMED — silently drops bad records
df_drop = (spark.read
    .format("json")
    .option("mode", "DROPMALFORMED")
    .schema(schema)
    .load("/Volumes/workspace/default/raw_transactions/bad_records.json"))

df_drop.write.format("delta").mode("overwrite").saveAsTable("temp_drop")

print("=== DROPMALFORMED MODE ===")
spark.sql("SELECT * FROM temp_drop").show(truncate=False)
print(f"Total records: {spark.sql('SELECT COUNT(*) FROM temp_drop').collect()[0][0]}")
print("Bob's bad record silently dropped!")

In [0]:
# FAILFAST — stops immediately on bad record
try:
    df_fail = (spark.read
        .format("json")
        .option("mode", "FAILFAST")
        .schema(schema)
        .load("/Volumes/workspace/default/raw_transactions/bad_records.json"))

    df_fail.write.format("delta").mode("overwrite").saveAsTable("temp_fail")
    print("No errors found!")

except Exception as e:
    print("=== FAILFAST MODE ===")
    print(f"❌ Pipeline failed as expected!")
    print(f"Error: {type(e).__name__}")